**IMPORTS NECESARIOS**

In [25]:
# Conduct necessary imports
import torch
import torch.nn as nn

**JENSEN DIVERGENCE**

In [196]:
# Implementation of the Kullback-Leibler divergence
def kl_divergence(p:torch.Tensor, q:torch.Tensor, eps:float = 1e-12)->torch.Tensor:
    """
        Implementation of the KL divergence

        KL(P||Q) = sum_x(P(x)*[logP(x) - logQ(x)])

        Params:
            p (torch.Tensor):
                The predicted distribution

            q (torch.Tensor):
                The target distribution 

            eps (float):
                Avoid ln(0) 

        Returns:
            kl (p|q) (torch.Tensor):
                The output tensor
    """

    # We state two probability distributions (P and Q)
    # P: the empiric distribution (N,|V|)
    print(f'    (kl) Aspect of the empiric distribution:{p.shape}')

    # Q: the ground truth distribution: peaked in 1 token (N,|V|)
    print(f'    (kl) Aspect of the gt distribution:{q.shape}')

    # Clamp the log  values -> avoid zeroes in log -> avoid modifiying original dist
    eps = 1e-12
    log_p = torch.log(torch.clamp(input = p, min = eps))
    log_q = torch.log(torch.clamp(input = q, min = eps))

    # Compute the KL
    # Expected shape (B, N) -> zero safety was previously ensured
    kl = (p*(log_p - log_q)).sum(-1)
    print(f'    (kl) Aspect of the KL distribution:{kl.shape}')
    return  kl


In [181]:
def jensen_divergence(p:torch.Tensor, q:torch.Tensor)->torch.Tensor:
    """
        Function aimed to compute the Jensen
        Divergence. The formula applied is the 
        following:

        JSD(P||Q) = 0.5*D(P||M) + 0.5*D(Q||M)

        where M is:M = (P + Q)*0.5

        Params:
            p(torch.Tensor):
                The predicted distribution
            q(torch.Tensor):
                The ground truth distribution
        
        Returns:
            jsd (float):
                Value for the JSD
    """
    # Get the mixture distribution M (B, N, |V|)
    M = 0.5*(p + q)
    print(f'(jsd) Aspect of the half distribution:', M.shape)

    # KL(P||M) -->(B, N, 1)
    print(f'(jsd) GETTING KL(P||M)')
    kl_p = kl_divergence(p = p, q = M)

    # KL(Q||M) --> (B, N, 1)
    print(f'(jsd) GETTING KL(Q||M)')
    kl_q = kl_divergence(p = q, q = M)


    # Get the final value  --> (B, N, 1)  
    jsd = 0.5*kl_p + 0.5*kl_q
    print(f'(jsd) Aspect of the jensen divergence:', jsd.shape)
    return jsd

In [189]:
# Define the distributions
b = 16
n = 100
v = 100
# 1. Predicted distribution 
p = torch.randn(b,n,v) 
# 2. Target distribution
q = torch.zeros(b,n,v, dtype = torch.float32)
q[:,:,0] = 1

In [194]:
# Get the KL
jsd = jensen_divergence(p, q)

(jsd) Aspect of the half distribution: torch.Size([16, 100, 100])
(jsd) GETTING KL(P||M)
    (kl) Aspect of the empiric distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the gt distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the KL distribution:torch.Size([16, 100])
(jsd) GETTING KL(Q||M)
    (kl) Aspect of the empiric distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the gt distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the KL distribution:torch.Size([16, 100])
(jsd) Aspect of the jensen divergence: torch.Size([16, 100])


**CLASS IMPLEMENTATION**

In [197]:
class JensenDivergence():    

    @staticmethod
    def kl_divergence(p:torch.Tensor, q:torch.Tensor, eps:float = 1e-12)->torch.Tensor:
        """
            Implementation of the KL divergence

            KL(P||Q) = sum_x(P(x)*[logP(x) - logQ(x)])

            Params:
                p (torch.Tensor):
                    The predicted distribution

                q (torch.Tensor):
                    The target distribution 

                eps (float):
                    Avoid ln(0) 

            Returns:
                kl (p|q) (torch.Tensor):
                    The output tensor
        """

        # We state two probability distributions (P and Q)
        # P: the empiric distribution (N,|V|)
        print(f'    (kl) Aspect of the empiric distribution:{p.shape}')

        # Q: the ground truth distribution: peaked in 1 token (N,|V|)
        print(f'    (kl) Aspect of the gt distribution:{q.shape}')

        # Clamp the log  values -> avoid zeroes in log -> avoid modifiying original dist
        eps = 1e-12
        log_p = torch.log(torch.clamp(input = p, min = eps))
        log_q = torch.log(torch.clamp(input = q, min = eps))

        # Compute the KL
        # Expected shape (B, N) -> zero safety was previously ensured
        kl = (p*(log_p - log_q)).sum(-1)
        print(f'    (kl) Aspect of the KL distribution:{kl.shape}')
        return  kl

    @staticmethod
    def __call__(p:torch.Tensor, q:torch.Tensor)->torch.Tensor:
        """
            Function aimed to compute the Jensen
            Divergence. The formula applied is the 
            following:

            JSD(P||Q) = 0.5*D(P||M) + 0.5*D(Q||M)

            where M is:M = (P + Q)*0.5

            Params:
                p(torch.Tensor):
                    The predicted distribution
                q(torch.Tensor):
                    The ground truth distribution
            
            Returns:
                jsd (float):
                    Value for the JSD
        """
        # Get the mixture distribution M (B, N, |V|)
        M = 0.5*(p + q)
        print(f'(jsd) Aspect of the half distribution:', M.shape)

        # KL(P||M) -->(B, N, 1)
        print(f'(jsd) GETTING KL(P||M)')
        kl_p = JensenDivergence.kl_divergence(p = p, q = M)

        # KL(Q||M) --> (B, N, 1)
        print(f'(jsd) GETTING KL(Q||M)')
        kl_q = JensenDivergence.kl_divergence(p = q, q = M)


        # Get the final value  --> (B, N, 1)  
        jsd = 0.5*kl_p + 0.5*kl_q
        print(f'(jsd) Aspect of the jensen divergence:', jsd.shape)
        return jsd

In [198]:
# Define the scorer
scorer = JensenDivergence()

# Forward it 
scorer(p, q)

(jsd) Aspect of the half distribution: torch.Size([16, 100, 100])
(jsd) GETTING KL(P||M)
    (kl) Aspect of the empiric distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the gt distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the KL distribution:torch.Size([16, 100])
(jsd) GETTING KL(Q||M)
    (kl) Aspect of the empiric distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the gt distribution:torch.Size([16, 100, 100])
    (kl) Aspect of the KL distribution:torch.Size([16, 100])
(jsd) Aspect of the jensen divergence: torch.Size([16, 100])


tensor([[14.9047, 25.5950, 14.8263,  ..., 25.8061, 23.4770, 24.5879],
        [15.5239, 10.6712, 17.4804,  ..., 19.7443, 16.8293, 20.1244],
        [27.7527, 29.2598, 16.6490,  ..., 26.1741, 18.6484, 13.8319],
        ...,
        [18.2580, 11.3702, 14.3118,  ..., 29.1040, 20.7870, 16.8431],
        [27.8074, 13.7663, 13.6159,  ..., 27.3448, 10.7434, 13.8251],
        [20.2509, 23.2286, 19.7273,  ..., 14.3985, 15.9961, 14.3742]])